# 82514 · Sesión S17 — Procesado clásico: de la foto de la mesa a la pieza en el robot

**Bloque 6** · lunes 16 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** La mesa del puesto de picking, vista por la cámara del curso, la de S16: la imagen como matriz de números; el umbral, que funciona con la luz del techo y se rompe con una lámpara, y el umbral adaptativo; las componentes conexas, los momentos y la circularidad; el centroide de la pieza, llevado a milímetros con la calibración (la de la cámara del curso o la que hizo en S16 uno de los dos) y a ángulos con la inversa de S10; la convolución, el suavizado, el gradiente, la sombra y Canny; y las esquinas de Harris.

**Se apoya en:** Corke (2023), caps. 11 y 12; Otsu (1979); Canny (1986); Harris y Stephens (1988); documentación de OpenCV.

**Cómo usarlo en clase.** Sigue el guion de la sesión S17 en las diapositivas del bloque 6. Ejecuta la celda de instalación una sola vez al empezar; en Colab tarda unos veinte segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos veinte segundos.
import subprocess, sys, importlib.metadata as md

OPENCV = '4.14.0.94'          # la versión de todo el bloque 6 (cv2.__version__ = '4.14.0')
FIJAS = {'scikit-image': '0.25.2'}               # las de Colab: se comprueban y, si cambian, se instalan antes de importarlas

def opencv_activo():
    """Versión de OpenCV que se importaría, mirada en otro proceso: Colab trae tres paquetes de OpenCV que
    comparten la carpeta cv2, y lo que dice pip no basta para saber cuál se carga."""
    r = subprocess.run([sys.executable, '-c', 'import cv2; print(cv2.__version__)'], capture_output=True, text=True)
    return r.stdout.strip()

def fijar():
    if opencv_activo() != OPENCV.rsplit('.', 1)[0]:
        print('Instalando OpenCV', OPENCV, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'opencv-python-headless=={OPENCV}'], check=False)
    for paquete, version in FIJAS.items():
        try:
            ya = md.version(paquete)
        except md.PackageNotFoundError:
            ya = None
        if ya != version:
            print('Instalando', paquete, version, '...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{paquete}=={version}'], check=False)

fijar()
import os, json, time, tempfile, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import cv2
warnings.filterwarnings('ignore')
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

def sin_ejes(*axs):
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])

print('OpenCV', cv2.__version__, '· numpy', np.__version__, '· en Colab:', EN_COLAB)
if cv2.__version__ != OPENCV.rsplit('.', 1)[0]:
    print('AVISO: este cuaderno se probó con OpenCV', OPENCV, '. Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar esta celda.')
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Ábrelo al empezar la sesión y ejecuta la primera celda: fija OpenCV 4.14.0, la versión de todo el bloque 6, en unos veinte segundos la primera vez (Colab trae otra).
>
> **En clase (nueve minutos):** el simulador y las secciones 1 a 3, con el ejercicio de clase, que se pone en común a la vuelta del descanso. Tened a mano vuestro fichero de S16. Al acabar, abre el cuaderno S17b y ejecuta su primera celda.
>
> **En casa:** la sección 4 (las esquinas) y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 0. La mesa del puesto de picking

*En clase, ejecuta estas celdas sin detenerte.*

La escena es la del bloque: el puesto de picking del almacén, con el 2R de B4 sobre su mesa y la cámara del curso **cenital, a 1 m**, en T_0C = Trans(0,3; 0,2; 1)·Rx(180°). La pieza es un disco de 40 mm de diámetro y 10 mm de alto en **P = (0,4; 0,2) m**, el punto de la inversa de S10; al lado, un cuadrado y un triángulo que no son nuestra pieza.

La foto no es de una cámara real: la fabrica un **simulador** con la física de la cámara sintética de S16. Cada píxel es un rayo que sale de la cámara con su K y su distorsión, se gira al mundo con la pose de la cámara y se corta con la mesa y con las piezas: la cara de arriba, a 10 mm; el lateral; y la sombra, si hay una lámpara. La mesa tiene la veta de la madera. Después vienen el desenfoque, el ruido del sensor (2 niveles) y los 8 bits. A cambio de no ser real, **se conoce la verdad** píxel a píxel: qué píxeles son pieza y dónde está cada una. Con eso se puede medir cuánto se equivoca cada método. La celda del simulador va plegada: ejecutadla sin abrirla. Lleva dentro, codificadas como en el cuaderno S16, las cinco unidades de los equipos, con su K, su distorsión y su montaje: es la verdad que se desvela el jueves 26, en S20.

Dos luces: **la del techo**, uniforme y sin sombras, y **una lámpara** a 30° sobre la mesa, a la derecha: la mesa pasa de la mitad de luz, a la izquierda, a toda la luz, a la derecha, y cada pieza echa su sombra.

In [ ]:
# @title El simulador de la mesa y de las cámaras de los equipos (ejecutadla sin abrirla)
# El simulador de la mesa del puesto de picking. Ejecútalo sin detenerte: es la cámara de S16 apuntando a la mesa.
# Cada píxel (submuestreado 2 x 2) es un rayo que sale de la cámara con su K y su distorsión, se gira al mundo con la
# pose de la cámara y se corta con la mesa y con las piezas. Después, desenfoque, ruido y 8 bits, como una foto.
W, H = 640, 480
F = 800.0
K_CURSO = np.array([[F, 0, W/2], [0, F, H/2], [0, 0, 1.0]])
DIST_CURSO = np.array([-0.25, 0.06, 0.0, 0.0, 0.0])
A1, A2 = 0.35, 0.20
TC = np.array([0.30, 0.20, 1.0])
R_0C = np.diag([1.0, -1.0, -1.0])            # Rx(180°): la camara mira hacia abajo
P_PIEZA = np.array([0.40, 0.20])
H_PIEZA, D_PIEZA = 0.010, 0.040

T_0C_NOMINAL = np.eye(4); T_0C_NOMINAL[:3, :3] = R_0C; T_0C_NOMINAL[:3, 3] = TC   # el plano del montaje


def T_C0_verdad():
    M = np.eye(4); M[:3, :3] = R_0C.T; M[:3, 3] = -R_0C.T @ TC
    return M


class Pieza:
    """Prisma recto de altura h: 'disco' (diametro d), 'cuadrado' (lado d) o 'triangulo' (lado d), girado ang."""
    def __init__(self, forma, x, y, d, ang=0.0, h=H_PIEZA, albedo=0.80):
        self.forma, self.c, self.d, self.ang, self.h, self.albedo = forma, np.array([x, y]), d, ang, h, albedo
        if forma == 'cuadrado':
            v = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]])*d/2
        elif forma == 'triangulo':
            r = d/np.sqrt(3)
            v = np.array([[r*np.cos(a), r*np.sin(a)] for a in (np.pi/2, np.pi/2 + 2*np.pi/3, np.pi/2 + 4*np.pi/3)])
        else:
            v = None
        if v is not None:
            ca, sa = np.cos(ang), np.sin(ang)
            self.v = v @ np.array([[ca, sa], [-sa, ca]]) + self.c

    def dentro(self, X, Y):
        if self.forma == 'disco':
            return (X - self.c[0])**2 + (Y - self.c[1])**2 <= (self.d/2)**2
        ok = np.ones(X.shape, bool)
        v = self.v
        for a, b in zip(v, np.roll(v, -1, axis=0)):      # poligono convexo, vertices en sentido antihorario
            ok &= (b[0] - a[0])*(Y - a[1]) - (b[1] - a[1])*(X - a[0]) >= 0
        return ok

    def area_px(self, f=F, Z=1.0):
        """Area aproximada de la cara de arriba en la imagen (a la distancia Z - h)."""
        e = f/(Z - self.h)
        if self.forma == 'disco':
            return np.pi*(self.d/2*e)**2
        if self.forma == 'cuadrado':
            return (self.d*e)**2
        return np.sqrt(3)/4*(self.d*e)**2


def vetas(X, Y):
    """La veta de la madera de la mesa: rayas onduladas a lo largo de y, cada 8 mm, de un 8 % de amplitud."""
    return 0.08*np.sin(2*np.pi*X/0.008 + 2.5*np.sin(2*np.pi*Y/0.15))


ESCENA = [Pieza('disco', 0.40, 0.20, 0.040),                 # la pieza: en P, el punto de la inversa de S10
          Pieza('cuadrado', 0.12, 0.34, 0.040, ang=0.35),     # dos piezas que no son la nuestra
          Pieza('triangulo', 0.56, 0.40, 0.050, ang=0.2)]


class Mesa:
    """La camara (K y distorsion de la unidad) a 1 m de la mesa, cenital: por defecto, en el plano del montaje,
    T_0C = Trans(0,3; 0,2; 1)·Rx(180°); con T_0C, donde de verdad la dejo su soporte (la unidad de S16)."""
    def __init__(self, K=K_CURSO, dist=DIST_CURSO, T_0C=None, ss=2):
        T = T_0C_NOMINAL if T_0C is None else np.asarray(T_0C, float)
        self.K, self.dist, self.ss = K, dist, ss
        self.R, self.o = T[:3, :3], T[:3, 3]                        # giro y origen de la camara en {0}
        us = (np.arange(W*ss) + 0.5)/ss - 0.5
        vs = (np.arange(H*ss) + 0.5)/ss - 0.5
        U, V = np.meshgrid(us, vs)
        pts = np.stack([U.ravel(), V.ravel()], -1).reshape(-1, 1, 2)
        n = cv2.undistortPointsIter(pts, K, dist, None, None,
                                    (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 60, 1e-12)).reshape(-1, 2)
        d = (self.R @ np.vstack([n.T, np.ones(len(n))])).T        # direccion del rayo en el mundo
        self.d = d
        self.X0, self.Y0 = self.corta(0.0)                         # donde corta cada rayo a la mesa

    def corta(self, z):
        lam = (z - self.o[2])/self.d[:, 2]
        return self.o[0] + lam*self.d[:, 0], self.o[1] + lam*self.d[:, 1]

    def foto(self, piezas=ESCENA, rng=None, luz=(1.0, 1.0), mesa=0.35, lateral=0.30, sombra=0.45,
             lampara=None, blur=0.7, ruido=2.0, textura=vetas, mascara=False):
        """luz = (en x = -0,1 m, en x = 0,7 m): la iluminacion de la mesa varia en linea recta a lo ancho.
        lampara: hacia donde esta la lampara, en metros de mesa por metro de altura (None: luz del techo, sin
        sombras; (1,73; 0,5), una lampara a 30° sobre la mesa, a la derecha). Devuelve la imagen
        en gris (uint8) y, si mascara, la verdad: un entero por pixel (0 mesa, i+1 la cara de arriba de la pieza i)."""
        rng = np.random.default_rng(0) if rng is None else rng
        X0, Y0 = self.X0, self.Y0
        L = luz[0] + (luz[1] - luz[0])*np.clip((X0 + 0.10)/0.80, 0, 1)
        alb = np.full(X0.shape, mesa)
        if textura is not None:
            alb = alb*(1 + textura(X0, Y0))
        verdad = np.zeros(X0.shape, np.int32)
        for i, p in enumerate(piezas):
            # solo los rayos que pasan cerca de la pieza (mas rapido: la escena entera son 1,2 millones de rayos)
            alc = p.d + (0 if lampara is None else np.hypot(*lampara)*p.h) + 0.01
            j = np.flatnonzero((X0 - p.c[0])**2 + (Y0 - p.c[1])**2 < alc**2)
            x0, y0 = X0[j], Y0[j]
            base = p.dentro(x0, y0)
            if lampara is not None:      # sombra: desde ese punto de la mesa, la lampara se ve a traves de la pieza
                somb = np.zeros(len(j), bool)
                for z in (0.25*p.h, 0.5*p.h, 0.75*p.h, p.h):
                    somb |= p.dentro(x0 + lampara[0]*z, y0 + lampara[1]*z)
                alb[j] = np.where(somb & ~base, alb[j]*sombra, alb[j])
            lam = (p.h - self.o[2])/self.d[j, 2]
            top = p.dentro(self.o[0] + lam*self.d[j, 0], self.o[1] + lam*self.d[j, 1])
            alb[j] = np.where(base & ~top, lateral, alb[j])     # el lateral: entra por el borde y no sale por la cara
            alb[j] = np.where(top, p.albedo, alb[j])
            verdad[j] = np.where(top, i + 1, verdad[j])
        img = 255*alb*L
        ss = self.ss
        img = img.reshape(H*ss, W*ss).reshape(H, ss, W, ss).mean(axis=(1, 3))
        img = cv2.GaussianBlur(img, (0, 0), blur)
        img = img + rng.normal(0, ruido, img.shape)
        img = np.clip(np.round(img), 0, 255).astype(np.uint8)
        if mascara:     # la verdad en la rejilla de pixeles: la pieza que cubre al menos 2 de los 4 subpixeles
            vs = verdad.reshape(H, ss, W, ss).transpose(0, 2, 1, 3).reshape(H, W, ss*ss)
            v = np.zeros((H, W), np.int32)
            for i in range(1, len(piezas) + 1):
                v[(vs == i).sum(-1)*2 >= ss*ss] = i
            return img, v
        return img


def a_la_mesa(uv, K, dist, T_C0, z=0.0):
    """Del pixel al plano z de {0}: quitar la distorsion, rayo K^-1·p, girarlo al mundo y cortarlo con el plano."""
    uv = np.atleast_2d(np.asarray(uv, float)).reshape(-1, 1, 2)
    n = cv2.undistortPointsIter(uv, K, np.asarray(dist, float), None, None,
                                (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 100, 1e-12)).reshape(-1, 2)
    n = np.column_stack([n, np.ones(len(n))])
    T = np.linalg.inv(np.asarray(T_C0, float))
    R, o = T[:3, :3], T[:3, 3]
    d = (R @ n.T).T
    lam = (z - o[2])/d[:, 2]
    return (o + lam[:, None]*d)[:, :2]


def ik2r(x, y, codo=+1):
    """La inversa cerrada del 2R de S10 (a1 = 0,35 m, a2 = 0,2 m); codo = +1, codo abajo."""
    c2 = (x*x + y*y - A1**2 - A2**2)/(2*A1*A2)
    q2 = codo*np.arccos(np.clip(c2, -1, 1))
    q1 = np.arctan2(y, x) - np.arctan2(A2*np.sin(q2), A1 + A2*np.cos(q2))
    return q1, q2


def piezas_en(mask, area_min=300, area_max=3000):
    """Componentes conexas de una mascara binaria: area, centroide (momentos), circularidad y caja."""
    n, et, st, cen = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for k in range(1, n):
        a = st[k, cv2.CC_STAT_AREA]
        if a < area_min or a > area_max:
            continue
        m = (et == k).astype(np.uint8)
        M = cv2.moments(m, binaryImage=True)
        cont = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0][0]
        per = cv2.arcLength(cont, True)
        x, y, w, h = st[k, :4]
        out.append(dict(area=M['m00'], u=M['m10']/M['m00'], v=M['m01']/M['m00'],
                        circ=4*np.pi*M['m00']/per**2, caja=(x, y, x + w, y + h)))
    return out

import zlib, base64
D2R = np.pi/180

# Las unidades de S16, con su montaje: la misma tabla codificada que el simulador del cuaderno S16 (sim_celda.py).
# Es la verdad que se desvela el jueves 26, en S20: no la decodifiquéis.
_U = 'eNo9kFkOwyAMRO+Sb8fCC0t6laonqXr3zji0ijCE4MebvI92PJ6rNW0SzurJejb1LlgNlnvcj/7HSw5D81yhjuapKR5drZonT1wsrQV3MLuknCFEJrcCiwWKU8Fcl4RhBwqxKU5A3+150/Cli8tZCDpibergBDmeFSUrytJZHIqYybbYUiFdqMBBpw5EMtAVeA8baHa/IERE3T63iu0ZyXD7kPsIS4rVn+kEzcoUBk1PB5ShqWH+y7KVhgA5S6RSoczX5wuk8U/j'
_UNIDADES = {int(k): v for k, v in json.loads(zlib.decompress(base64.b64decode(_U))).items()}


def _Rx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def _Ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def _Rz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def unidad(e):
    """La camara que hace las fotos: K, distorsion y T_0C de la unidad e (0, la del curso, en el plano del montaje),
    construidos como Camara.__init__ del simulador de S16."""
    fx, cx, cy, k1, k2, p1, p2, dx, dy, dz, ax, ay, az = _UNIDADES[e]
    M = np.eye(4); M[:3, :3] = _Rz(az*D2R) @ _Ry(ay*D2R) @ _Rx(ax*D2R); M[:3, 3] = np.array([dx, dy, dz])*1e-3
    return np.array([[fx, 0, cx], [0, fx, cy], [0, 0, 1.0]]), np.array([k1, k2, p1, p2, 0.0]), T_0C_NOMINAL @ M

**¿Con qué cámara?** Con `EQUIPO = 0`, la foto la hace la cámara del curso (f = 800 px, k₁ = −0,25, k₂ = 0,06), montada exactamente donde dice el plano, y se mide con su K, que se conoce exacta. Con `EQUIPO` de 1 a 5, la foto la hace **la unidad de vuestro equipo**, la que calibrasteis en S16, con su K, su distorsión y su montaje de verdad, y el cuaderno mide con **vuestra calibración**: el fichero `calibracion_E⟨n⟩⟨letra⟩.json` que guardó en S16 uno de los dos, el que elijáis (está también en su entrega de la Tarea de Moodle). Poned en `LETRA` la letra de ese fichero, de la A a la E. En Colab, la celda os pide el fichero. Si no lo tenéis, mide **sin calibrar**: con la hoja de datos (800 px, sin distorsión) y el plano del montaje, y lo avisa. Al cambiar `EQUIPO` o `LETRA`, volved a ejecutar desde esta celda.

In [ ]:
EQUIPO = 0          # 0: la cámara del curso. De 1 a 5: la unidad de vuestro equipo, la de S16
LETRA = 'A'         # de 'A' a 'E': la letra del JSON de S16 que cargáis, el de cualquiera de los dos

def _clave(d, *nombres):
    for n in nombres:
        if d.get(n) is not None:
            return d[n]
    return None

def calibracion(e, letra):
    """Con qué se mide: (K, distorsión, T_C0). La de la cámara del curso, exacta; o la de un JSON de S16, el de uno de los dos."""
    if e == 0:
        return K_CURSO, DIST_CURSO, T_C0_verdad()
    assert letra in list('ABCDE'), "LETRA es la del fichero de S16: una de 'A', 'B', 'C', 'D' o 'E', entre comillas"
    nombre = f'calibracion_E{e}{letra}.json'
    if not os.path.exists(nombre) and EN_COLAB:
        from google.colab import files
        print('Sube', nombre, '(el que guardó en S16 uno de los dos; también está en su entrega de Moodle):')
        subidos = files.upload()
        if subidos:
            nombre = next(iter(subidos))          # el navegador puede haberlo llamado «… (1).json»
    if os.path.exists(nombre):
        d = json.load(open(nombre, encoding='utf-8'))
        ident = f"E{d.get('equipo', e)}{str(d.get('pareja', letra)).strip().upper()}"   # 'pareja' guarda la letra
        if ident != f'E{e}{letra}':
            print(f'AVISO: {nombre} es la calibración de {ident}, no la de E{e}{letra}: poned en EQUIPO y LETRA '
                  'los de ese fichero y volved a ejecutar desde esta celda.')
        K = np.array(_clave(d, 'K', 'matriz_K'), float)
        dist = np.array(_clave(d, 'distorsion_k1_k2_p1_p2_k3', 'dist', 'distorsion'), float).ravel()
        T = np.array(_clave(d, 'T_C0', 'extrinsecos'), float)
        if K.shape == (3, 3) and dist.size in (4, 5, 8, 12, 14) and T.shape == (4, 4) and np.isfinite(dist).all():
            rms = _clave(d, 'rms_px', 'rms')
            print(f'Calibración de {ident} ({nombre}): fx = {coma(K[0, 0], 1)} px, RMS = {coma(rms, 3) if rms is not None else "?"} px')
            return K, dist, T
        print('AVISO:', nombre, 'no trae K (3 x 3), la distorsión y T_C0 (4 x 4).')
    print('AVISO: sin calibración: se mide con la hoja de datos (800 px, sin distorsión) y el plano del montaje.')
    return K_CURSO, np.zeros(5), T_C0_verdad()

cam = Mesa(*unidad(EQUIPO))                   # la cámara que hace las fotos (su verdad no se imprime)
CAL = calibracion(EQUIPO, LETRA)              # con qué se mide
LAMPARA = dict(luz=(0.5, 1.0), lampara=(1.73, 0.5))
I_t, V_t = cam.foto(mascara=True)                                  # la luz del techo
I_l, V_l = cam.foto(rng=np.random.default_rng(1), mascara=True, **LAMPARA)   # con la lámpara

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
axs[0].imshow(I_t, cmap='gray', vmin=0, vmax=255); axs[0].set_title('La luz del techo')
axs[1].imshow(I_l, cmap='gray', vmin=0, vmax=255); axs[1].set_title('Con una lámpara a la derecha')
sin_ejes(*axs); plt.tight_layout(); plt.show()

## 1. Una matriz de números, y el umbral

*En clase. Son las cuentas de las diapositivas 23 y 24.*

Para el ordenador, la foto es una **matriz de números**: en gris, uno por píxel, de 0 (negro) a 255 (blanco). `I[v, u]`: la fila es v, hacia abajo, y la columna es u, hacia la derecha, como el píxel de S16. 640 × 480 son 307 200 píxeles; en color, tres números por píxel.

**El umbral** separa la pieza de la mesa: blanco si I > T, negro si no. Para elegir T se mira el **histograma**, cuántos píxeles hay de cada gris. **Otsu** (1979) elige la T que mejor separa sus dos montañas: la que deja cada grupo lo más apretado posible (la que hace máxima la varianza entre los dos grupos).

**El umbral adaptativo** compara cada píxel con la media de su vecindario (aquí, 51 × 51 píxeles) más un margen (10 niveles): si la luz cambia despacio a lo ancho de la mesa, el vecindario cambia con ella. Deja motas sueltas, por la veta; una **apertura** de 3 × 3 las borra (sección 2).

In [ ]:
print('La foto:', I_t.shape, I_t.dtype, '; la mesa ronda', int(np.median(I_t[V_t == 0])), 'y la cara del disco', int(np.median(I_t[V_t == 1])))
vd, ud = np.argwhere(V_t == 1).mean(0).round().astype(int)        # el centro del disco, en la verdad del simulador
v0, u0 = vd - 3, ud - 21                                          # 6 x 6 en su borde izquierdo (237 y 380 con la cámara del curso)
print(f'6 x 6 píxeles del borde izquierdo del disco (filas {v0} a {v0 + 5}, columnas {u0} a {u0 + 5}):')
print(I_t[v0:v0 + 6, u0:u0 + 6])

def encontradas(mask, V):
    """Cuántas piezas verdaderas cubre una componente de la máscara (IoU de las cajas de 0,5 o más)."""
    ps = piezas_en(mask, area_min=50, area_max=5000)
    def caja(k):
        ij = np.argwhere(V == k); return (ij[:, 1].min(), ij[:, 0].min(), ij[:, 1].max() + 1, ij[:, 0].max() + 1)
    def iou(A, B):
        ix = max(0, min(A[2], B[2]) - max(A[0], B[0])); iy = max(0, min(A[3], B[3]) - max(A[1], B[1]))
        return ix*iy/((A[2]-A[0])*(A[3]-A[1]) + (B[2]-B[0])*(B[3]-B[1]) - ix*iy)
    return sum(any(iou(p['caja'], caja(k)) >= 0.5 for p in ps) for k in range(1, V.max() + 1))

mal = lambda b, V: 100*((b > 0) != (V > 0)).mean()      # % de píxeles mal clasificados frente a la verdad

th_t, b_t = cv2.threshold(I_t, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f'\nLuz del techo: Otsu elige {th_t:.0f}; mal clasificados {coma(mal(b_t, V_t))} %; piezas {encontradas(b_t, V_t)} de 3')
th_l, b_l = cv2.threshold(I_l, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f'Con la lámpara, la mesa va de {np.percentile(I_l[V_l == 0], 1):.0f} a {np.percentile(I_l[V_l == 0], 99):.0f}:')
print(f'   Otsu elige {th_l:.0f}; mal {coma(mal(b_l, V_l), 1)} %; {cv2.connectedComponents(b_l)[0] - 1} trozos; piezas {encontradas(b_l, V_l)} de 3')
_, b_f = cv2.threshold(I_l, th_t, 255, cv2.THRESH_BINARY)
print(f'   el umbral de antes ({th_t:.0f}): mal {coma(mal(b_f, V_l))} %; piezas {encontradas(b_f, V_l)} de 3 (el cuadrado queda en {np.median(I_l[V_l == 2]):.0f})')
b_a0 = cv2.adaptiveThreshold(I_l, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 51, -10)
b_a = cv2.morphologyEx(b_a0, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
print(f'   adaptativo (51 x 51, más 10) y apertura de 3 x 3: mal {coma(mal(b_a, V_l))} %; piezas {encontradas(b_a, V_l)} de 3')

fig, axs = plt.subplots(1, 4, figsize=(15, 3.4))
axs[0].bar(np.arange(256), np.bincount(I_l.ravel(), minlength=256), width=1, color=IQS_AZUL); axs[0].set_yscale('log')
axs[0].axvline(th_l, color=ROJO); axs[0].axvline(th_t, color=AMBAR, ls='--'); axs[0].set_title('Histograma, con la lámpara'); ejes(axs[0])
for ax, (b, t) in zip(axs[1:], ((b_l, f'Otsu ({th_l:.0f})'), (b_f, f'El de antes ({th_t:.0f})'), (b_a, 'Adaptativo y apertura'))):
    ax.imshow(b, cmap='gray'); ax.set_title(t); sin_ejes(ax)
plt.tight_layout(); plt.show()

## 2. De la máscara a la pieza, y de la pieza al robot

*En clase, con el ejercicio de clase. Son las cuentas de las diapositivas 25 y 26.*

**Las componentes conexas** ponen una etiqueta a cada grupo de píxeles blancos que se tocan (`cv2.connectedComponentsWithStats`). **Los momentos** son sumas sobre los píxeles de cada pieza: m_pq = Σ u^p·v^q. m₀₀ cuenta los píxeles, el área; m₁₀/m₀₀ y m₀₁/m₀₀ son la media de u y de v, **el centroide**. **La circularidad**, 4π·área/perímetro², vale 1 en un círculo, 0,785 en un cuadrado y 0,605 en un triángulo equilátero: separa el disco de las otras piezas.

**Del centroide a la mesa**, con lo de S16: se quita la distorsión, el píxel da un rayo (K⁻¹·p̃), se gira al mundo con la pose de la cámara y se corta con un plano. ¿Con cuál? La cara de arriba del disco está a **10 mm** de la mesa: es lo que la cámara ve. **De la mesa al robot**, con la inversa cerrada del 2R de S10 (a₁ = 0,35 m, a₂ = 0,2 m), con el codo abajo.

In [ ]:
# las motas de la veta, antes y después de la apertura
for margen in (8, 10):
    b0 = cv2.adaptiveThreshold(I_l, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 51, -margen)
    n0 = cv2.connectedComponents(b0)[0] - 1
    n1 = cv2.connectedComponents(cv2.morphologyEx(b0, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)))[0] - 1
    print(f'adaptativo más {margen}: {n0} componentes; tras la apertura de 3 x 3, {n1}')

nombres = {1: 'disco', 2: 'cuadrado', 3: 'triángulo'}
print('\nCon la luz del techo y Otsu, pieza a pieza:')
for p in piezas_en(b_t):
    k = V_t[int(round(p['v'])), int(round(p['u']))]
    print(f"  {nombres.get(k, '?'):10s} área m00 = {p['area']:5.0f} px ; centroide ({coma(p['u'])}; {coma(p['v'])}) ; circularidad {coma(p['circ'], 3)}")

# los momentos, a mano, para el disco: sumas sobre sus píxeles
disco = [p for p in piezas_en(b_t) if p['circ'] > 0.85 and 500 < p['area'] < 1200][0]
_, et = cv2.connectedComponents(b_t)
k = et[int(round(disco['v'])), int(round(disco['u']))]
v, u = np.nonzero(et == k)
m00, m10, m01 = len(u), u.sum(), v.sum()
print(f'\nA mano: m00 = {m00}, m10/m00 = {coma(m10/m00)}, m01/m00 = {coma(m01/m00)}  (cv2.moments da lo mismo)')
uc, vc = disco['u'], disco['v']

In [ ]:
# del centroide a la mesa y al robot (CAL = (K, distorsión, T_C0): la de la cámara del curso o la de uno de los dos en S16)
Pm = a_la_mesa((uc, vc), *CAL, z=H_PIEZA)[0]                      # cortando el rayo con la cara, a 10 mm
q1, q2 = ik2r(*Pm, +1)
print(f'El disco, en la mesa: ({coma(Pm[0], 4)}; {coma(Pm[1], 4)}) m, a {coma(1e3*np.linalg.norm(Pm - P_PIEZA))} mm de P')
print(f'q (codo abajo) = ({coma(np.degrees(q1))}°; {coma(np.degrees(q2))}°)')
# lo que cuesta no quitar la distorsión (solo con la K de la cámara)
Psd = a_la_mesa((uc, vc), CAL[0], np.zeros(5), CAL[2], z=H_PIEZA)[0]
print(f'Sin quitar la distorsión: a {coma(1e3*np.linalg.norm(Psd - P_PIEZA))} mm de P')
# el ruido del centroide: 30 fotos iguales con distinto ruido del sensor
cs = []
for s in range(30):
    I_ = cam.foto(rng=np.random.default_rng(1000 + s))
    _, b_ = cv2.threshold(I_, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    d_ = [p for p in piezas_en(b_) if p['circ'] > 0.85][0]
    cs.append((d_['u'], d_['v']))
print(f'El centroide tiembla σ = ({coma(np.std(cs, 0)[0], 3)}; {coma(np.std(cs, 0)[1], 3)}) px de foto a foto')

### Ejercicio de clase (en clase; se pone en común a la vuelta del descanso)

El píxel de la pieza ya no viene dado, como en S16: sale de segmentar. **Con la lámpara, ¿a cuántos milímetros de P sale el disco si se segmenta con Otsu, con el umbral de antes (el de Otsu con la luz del techo) y con el adaptativo?** Compáralo con el ruido del centroide de la celda anterior. Si tenéis el fichero de S16 de uno de los dos, poned `EQUIPO` y `LETRA`, volved a ejecutar desde la celda de la cámara y mirad también la última línea: la pieza con vuestra calibración y **sin calibrar** (la hoja de datos y el plano del montaje).

In [ ]:
# Ejercicio de clase: lo que cuesta segmentar mal, con la lámpara
def el_disco(b):
    """La componente que pasa la regla del disco: circularidad mayor que 0,85 y área de 500 a 1200 píxeles."""
    d = [p for p in piezas_en(b) if p['circ'] > 0.85 and 500 < p['area'] < 1200]
    return d[0] if d else None

for nom, b in ((f'Otsu ({th_l:.0f})', b_l), (f'el umbral de antes ({th_t:.0f})', b_f), ('el adaptativo', b_a)):
    d = el_disco(b)
    if d is None:
        print(f'{nom}: ninguna componente pasa la regla: el disco no aparece')
        continue
    Pq = a_la_mesa((d['u'], d['v']), *CAL, z=H_PIEZA)[0]
    print(f"{nom}: centroide ({coma(d['u'])}; {coma(d['v'])}), {d['area']:.0f} px ; la pieza a {coma(1e3*np.linalg.norm(Pq - P_PIEZA))} mm de P")
Psc = a_la_mesa((uc, vc), K_CURSO, np.zeros(5), T_C0_verdad(), z=H_PIEZA)[0]   # sin calibrar: hoja de datos y plano del montaje
con = 'la cámara del curso' if EQUIPO == 0 else ('vuestra calibración' if CAL[0] is not K_CURSO else 'la hoja de datos (no hay calibración)')
print(f'Con la luz del techo: con {con}, a {coma(1e3*np.linalg.norm(Pm - P_PIEZA))} mm ;'
      f' sin calibrar, a {coma(1e3*np.linalg.norm(Psc - P_PIEZA), 2 if np.linalg.norm(Psc - P_PIEZA) < 1e-3 else 1)} mm')

## 3. La convolución, los bordes y Canny

*En clase. Son las cuentas de las diapositivas 27 a 30, con la foto de la cámara del curso, sea cual sea `EQUIPO`.*

**La convolución**: un núcleo w de 3 × 3 recorre la imagen y cada píxel de salida es la suma de sus vecinos, cada uno por su peso, O(u, v) = Σ w(i, j)·I(u + i, v + j). Los núcleos que **suavizan** suman 1 (la caja; la gaussiana, con una anchura σ en píxeles); los que **derivan** suman 0 (Sobel en u resta la columna izquierda de la derecha, con la del centro el doble). `cv2.filter2D` aplica el núcleo sin voltearlo: en rigor, es una correlación, como la capa de una red convolucional.

**El gradiente**: Sobel en u y en v dan I_u e I_v; la magnitud es ‖∇I‖ = √(I_u² + I_v²) y la dirección, atan2(I_v, I_u). Aquí se divide entre 8, para que salga en niveles por píxel. **Canny** (1986) suaviza con σ, calcula el gradiente, se queda con el máximo a través de cada borde (supresión de no máximos) y une con dos umbrales (histéresis). `cv2.Canny` usa Sobel sin dividir entre 8: los umbrales de 3 y 8 niveles por píxel son 24 y 64.

In [ ]:
if EQUIPO != 0:      # las secciones 3 y 4 son las cuentas de las diapositivas 27 a 31: siempre con la foto de la cámara del curso
    cam0 = Mesa()
    I_t, V_t = cam0.foto(mascara=True)
    I_l, V_l = cam0.foto(rng=np.random.default_rng(1), mascara=True, **LAMPARA)
    _, b_t = cv2.threshold(I_t, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    uc, vc = [(p['u'], p['v']) for p in piezas_en(b_t) if p['circ'] > 0.85][0]
    print('Secciones 3 y 4: con la foto de la cámara del curso, la de las diapositivas (vuestra unidad, en las secciones 1 y 2).')
SOBEL = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
ub, vb = int(round(uc)) - 17, int(round(vc)) - 1          # el borde izquierdo del disco (384; 239 con la cámara del curso)
P3 = I_t[vb-1:vb+2, ub-1:ub+2].astype(int)
print(f'3 x 3 píxeles del borde del disco, alrededor de ({ub}; {vb}):\n', P3)
print('con Sobel en u, a mano:', int((SOBEL*P3).sum()), '; entre 8:', coma((SOBEL*P3).sum()/8, 1), 'niveles por píxel')
print('cv2.filter2D en ese píxel:', coma(cv2.filter2D(I_t.astype(float), -1, SOBEL)[vb, ub], 0))
print('sumas: caja', coma(np.full((3, 3), 1/9).sum(), 0), '; gaussiana', coma((np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]])/16).sum(), 0), '; Sobel', coma(SOBEL.sum(), 0))

def anchura_borde(J, v=240, u0=376, u1=392):
    """Del 10 al 90 % del salto del borde izquierdo del disco, en píxeles."""
    pr = J[v, u0:u1].astype(float); lo, hi = pr[:3].mean(), pr[-3:].mean()
    x = np.arange(len(pr))
    return np.interp(lo + 0.9*(hi - lo), pr, x) - np.interp(lo + 0.1*(hi - lo), pr, x)

zona = (slice(330, 400), slice(200, 300))                     # un trozo de mesa sin piezas
print('\nSuavizar: el gris de la mesa (ruido y veta) y la anchura del borde')
for nom, J in (('sin suavizar', I_t.astype(float)), ('gaussiana σ = 1', cv2.GaussianBlur(I_t.astype(float), (0, 0), 1)),
               ('gaussiana σ = 2', cv2.GaussianBlur(I_t.astype(float), (0, 0), 2)), ('gaussiana σ = 4', cv2.GaussianBlur(I_t.astype(float), (0, 0), 4)),
               ('caja de 7 x 7', cv2.blur(I_t.astype(float), (7, 7)))):
    print(f'  {nom:16s} σ de la mesa {coma(J[zona].std())} niveles ; borde de {coma(anchura_borde(J), 1)} px')

def gradiente(I, s=1.0):
    J = cv2.GaussianBlur(I.astype(float), (0, 0), s) if s > 0 else I.astype(float)
    Iu, Iv = cv2.Sobel(J, cv2.CV_64F, 1, 0)/8, cv2.Sobel(J, cv2.CV_64F, 0, 1)/8
    return Iu, Iv, np.hypot(Iu, Iv)

_, _, G_l = gradiente(I_l)
print('\nCon la lámpara, fila 240: el gris y ‖∇I‖ de u = 365 a 389')
print(' ', I_l[240, 365:390])
print(' ', np.round(G_l[240, 365:390]).astype(int))
print('  la sombra da un borde de', coma(G_l[240, 365:378].max(), 1), 'niveles por píxel; el disco,', coma(G_l[240, 378:390].max(), 1),
      '; la mesa, como mucho', coma(G_l[zona].max(), 1))

In [ ]:
lejos = cv2.dilate((V_l > 0).astype(np.uint8), np.ones((31, 31), np.uint8)) == 0   # la mesa, lejos de las piezas
cannys = {}
for s in (0, 1, 2, 4):
    J = cv2.GaussianBlur(I_l, (0, 0), s) if s else I_l
    cannys[s] = cv2.Canny(J, 3*8, 8*8, L2gradient=True)
    print(f'Canny con σ = {s}: {int((cannys[s] > 0).sum())} px de borde; {int((cannys[s][lejos] > 0).sum())} en la mesa, lejos de las piezas')
_, _, G1 = gradiente(I_l, 1)
print('‖∇I‖ > 3 con σ = 1, sin Canny:', int((G1 > 3).sum()), 'px, en bandas anchas')

fig, axs = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (t, b) in zip(axs, (('‖∇I‖ > 3', (G1 > 3)*255), ('Canny, σ = 0', cannys[0]), ('Canny, σ = 1', cannys[1]), ('Canny, σ = 4', cannys[4]))):
    ax.imshow(255 - b[60:480, 100:640], cmap='gray'); ax.set_title(t); sin_ejes(ax)
plt.tight_layout(); plt.show()

## 4. Las esquinas: Harris (en casa)

Un borde no sirve para reencontrar un punto: a lo largo de él todo se parece. Una **esquina** tiene dos direcciones de gradiente. En una ventana, la **matriz de estructura** M = Σ [[I_u², I_u·I_v], [I_u·I_v, I_v²]] lo mide: en una zona plana sus dos autovalores son pequeños; en un borde, uno es grande; en una esquina, los dos. Harris y Stephens (1988) usan c = det M − 0,04·(tr M)², sin calcular autovalores: c ≈ 0 en lo plano, negativa en un borde, positiva en una esquina.

`cv2.goodFeaturesToTrack(..., useHarrisDetector=True)` busca los máximos de c y los separa una distancia mínima. Las esquinas se **reencuentran** al girar la foto, pero no al cambiar de escala con la misma ventana. Las 54 esquinas del tablero de S16 eran de este tipo; con una firma de su entorno (el descriptor), se emparejan entre vistas: es la base del SLAM visual (S19, el lunes 23).

In [ ]:
Js = cv2.GaussianBlur(I_t, (0, 0), 1).astype(np.float32)
Ix, Iy = cv2.Sobel(Js, cv2.CV_64F, 1, 0)/8, cv2.Sobel(Js, cv2.CV_64F, 0, 1)/8

def M_est(u, v, w=3):
    s = (slice(v - w, v + w + 1), slice(u - w, u + w + 1)); a, b = Ix[s], Iy[s]
    return np.array([[(a*a).sum(), (a*b).sum()], [(a*b).sum(), (b*b).sum()]])

for nom, (u, v) in (('la mesa', (250, 380)), ('el borde del disco', (385, 240)), ('la esquina del cuadrado', (157, 119))):
    M = M_est(u, v); lam = np.linalg.eigvalsh(M); c = np.linalg.det(M) - 0.04*np.trace(M)**2
    print(f'{nom:24s} autovalores {lam[0]:9.0f} y {lam[1]:9.0f} ; c = {coma(c, 0)}')

def esquinas(I, n=50):
    q = cv2.goodFeaturesToTrack(I, maxCorners=n, qualityLevel=0.05, minDistance=8, blockSize=7, useHarrisDetector=True, k=0.04)
    return np.zeros((0, 2)) if q is None else q.reshape(-1, 2)

E = esquinas(I_t)
Mr = cv2.getRotationMatrix2D((W/2, H/2), 30, 1.0)
Ir = cv2.warpAffine(I_t, Mr, (W, H), borderMode=cv2.BORDER_REPLICATE)
Er = esquinas(Ir)
proy = (Mr[:, :2] @ E.T).T + Mr[:, 2]
dentro = (proy[:, 0] > 5) & (proy[:, 0] < W - 5) & (proy[:, 1] > 5) & (proy[:, 1] < H - 5)
d = np.linalg.norm(proy[dentro][:, None] - Er[None], axis=2).min(1)
print(f'\n{len(E)} esquinas; girando la foto 30°, de las {dentro.sum()} que siguen dentro se reencuentran {(d < 2).sum()} (a menos de 2 px)')
Es = esquinas(cv2.resize(I_t, (W//2, H//2), interpolation=cv2.INTER_AREA))*2 + 0.5
print(f'A la mitad de tamaño, con la misma ventana: {len(Es)} esquinas, {(np.hypot(Es[:, 0] - uc, Es[:, 1] - vc) < 25).sum()} en el disco, que no tiene')

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
axs[0].imshow(I_t, cmap='gray'); axs[0].plot(E[:, 0], E[:, 1], 'o', mfc='none', mec=ROJO, ms=12, mew=2); axs[0].set_title('Harris')
axs[1].imshow(Ir, cmap='gray'); axs[1].plot(Er[:, 0], Er[:, 1], 'o', mfc='none', mec=IQS_VERDE, ms=12, mew=2); axs[1].set_title('Girada 30°')
sin_ejes(*axs); plt.tight_layout(); plt.show()

### Ejercicio 1 (en casa): los dos mandos de Canny

Con σ = 1 px, prueba los umbrales (1 y 3), (3 y 8) y (8 y 20) niveles por píxel (en `cv2.Canny`, por 8). Cuenta los píxeles de borde en total y en la mesa lejos de las piezas. Compáralo con lo que hace σ. ¿Qué mando decide **a qué escala** se mira y cuál **cuánta evidencia** se exige?

In [ ]:
# Ejercicio 1: prueba aquí
# J = cv2.GaussianBlur(I_l, (0, 0), 1)
# for bajo, alto in ((1, 3), (3, 8), (8, 20)):
#     b = cv2.Canny(J, bajo*8, alto*8, L2gradient=True)
#     print(bajo, alto, int((b > 0).sum()), int((b[lejos] > 0).sum()))

### Ejercicio 2 (en casa): una foto real

`skimage.data.coins()` es una foto real de 24 monedas sobre una mesa, con más luz arriba que abajo. Aplica Otsu y el umbral adaptativo con un vecindario de 51 × 51 (margen 10, como en la mesa del puesto) y de 101 × 101 (margen 5), los tres con una apertura de 3 × 3. Cuenta las componentes de 500 píxeles o más y, de ellas, las **monedas enteras**: redondas (circularidad mayor que 0,8) y de menos de 2600 píxeles. Mira las máscaras. ¿Cuál gana, y por qué el vecindario de 51 × 51, que en la mesa iba bien, aquí pierde?

In [ ]:
# Ejercicio 2: prueba aquí
# from skimage import data
# M = data.coins()
# def cuenta(b):
#     b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
#     n, et, st, _ = cv2.connectedComponentsWithStats(b)
#     grandes = [k for k in range(1, n) if st[k, cv2.CC_STAT_AREA] >= 500]
#     enteras = [k for k in grandes if st[k, cv2.CC_STAT_AREA] < 2600 and piezas_en(((et == k)*255).astype(np.uint8), 500, 2600)[0]['circ'] > 0.8]
#     return len(grandes), len(enteras)
# _, bo = cv2.threshold(M, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
# print('Otsu', cuenta(bo))
# for bloque, margen in ((51, 10), (101, 5)):
#     print(bloque, cuenta(cv2.adaptiveThreshold(M, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, bloque, -margen)))

### Ejercicio 3 (en casa): la pieza en la esquina del campo

Pon el disco en (0,65; 0,45) m, cerca de la esquina de la imagen: `ESCENA_ESQ = [Pieza('disco', 0.65, 0.45, 0.040)]` y `cam.foto(ESCENA_ESQ, mascara=True)`. Con la cámara del curso (`EQUIPO = 0`), calcula a cuántos milímetros de su sitio sale cortando con la cara, cortando con la mesa y sin quitar la distorsión. Compáralo con lo que salía en P. ¿Qué error crece con la distancia al centro de la imagen y cuál no?

In [ ]:
# Ejercicio 3: prueba aquí
# ESCENA_ESQ = [Pieza('disco', 0.65, 0.45, 0.040)]
# I3, V3 = cam.foto(ESCENA_ESQ, mascara=True)
# _, b3 = cv2.threshold(I3, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
# p3 = piezas_en(b3)[0]
# for dist, z in ((CAL[1], H_PIEZA), (CAL[1], 0.0), (np.zeros(5), H_PIEZA)):
#     Q = a_la_mesa((p3['u'], p3['v']), CAL[0], dist, CAL[2], z=z)[0]
#     print(coma(1e3*np.linalg.norm(Q - np.array([0.65, 0.45]))))

---

## Soluciones

**Ejercicio de clase.** Con la cámara del curso y la lámpara: con Otsu (68), ninguna componente pasa la regla del disco: Otsu parte la mesa y el disco se une a su mitad clara. Con el umbral de antes (144), el disco sale, pero cortado por el lado oscuro (754 px frente a 813): el centroide se va 0,21 px y la pieza, a 0,25 mm de P. Con el adaptativo, a 0,03 mm, casi como con la luz del techo (0,02 mm). El ruido del centroide es de unas 0,03 px, unas 0,03 mm: un umbral mal puesto mueve la pieza unas ocho veces más, y promediando fotos no se quita, porque es sistemático; se quita segmentando bien (comparando con el entorno) o controlando la luz. Con una calibración de S16, la pieza sale a menos de medio milímetro (de 0,06 a 0,20 mm con la tabla de vistas mejorada; de 0,23 a 0,45 con la de partida). Sin calibrar, con la hoja de datos y el plano del montaje, de 9 a 18 mm según la unidad: la unidad no es la hoja de datos, y sobre todo no está donde dice el plano. Con `EQUIPO = 0` la línea «sin calibrar» solo quita la distorsión (0,27 mm), porque la cámara del curso sí es la de la hoja de datos y está donde dice el plano.

**Ejercicio 1.** Con σ = 1: los umbrales (1 y 3) dan 44 348 px de borde, 42 695 en la mesa lejos de las piezas; (3 y 8), 560 y 0; (8 y 20), 366 y 0. Bajar los umbrales deja pasar la veta y el ruido; subirlos pierde los bordes débiles, como el de la sombra. σ decide qué tamaño de detalle cuenta como borde (la escala); los umbrales, cuánta evidencia se exige. Son mandos distintos: subir los umbrales para quitar ruido cuando el problema era la escala es el error habitual.

**Ejercicio 2.** Otsu da 25 componentes de 500 px o más, y 14 monedas enteras: arriba, con más luz, el fondo pasa del umbral y se une a las monedas de la primera fila (una componente de más de 6600 píxeles), y las monedas más oscuras se agujerean. El adaptativo de 51 × 51 da 24 componentes, pero solo 9 enteras: el vecindario es del tamaño de una moneda (unos 40 px), y dentro de una moneda clara la media también es alta, así que el centro queda por debajo y la moneda se agujerea. Con 101 × 101, 19 enteras. Moraleja: el vecindario tiene que ser bastante mayor que la pieza, como en la mesa del puesto (51 px frente a discos de 32 px); y una foto real nunca es tan limpia como la sintética.

**Ejercicio 3.** En (0,65; 0,45), a 430 mm del eje óptico: cortando con la cara, 0,09 mm; con la mesa, 4,26 mm (ℓ·h/(Z − h) = 430·10/990 = 4,3 mm, la cuenta de S16); sin quitar la distorsión, 19,5 mm. Los dos errores sistemáticos crecen con la distancia al centro de la imagen: el del espesor, en línea recta; el de la distorsión, mucho más deprisa (con r³). En P, cerca del centro, los dos eran pequeños; en la esquina, no se pueden ignorar.

---

## Para llevarse de esta sesión

- **Una imagen es una matriz de números**, y el procesado clásico son cuentas con ellos: umbral, componentes, momentos, convoluciones.
- **El umbral global supone la misma luz en toda la mesa.** Con una lámpara, Otsu parte la mesa; el adaptativo compara cada píxel con su entorno.
- **El centroide es un píxel**, y sale de segmentar: K, la distorsión y la pose de S16 lo llevan a la mesa, cortando con el plano de la cara, y la inversa de S10, al robot. Lo que pesa es lo sistemático (la segmentación, la calibración), no el ruido.
- **Una convolución, muchos núcleos**: suavizar suma 1, derivar suma 0. El gradiente marca la intensidad, no la materia: la sombra también da borde.
- **σ decide la escala** de Canny; los umbrales, la evidencia. **Las esquinas se reencuentran** al girar, no al cambiar de escala.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*